# Cliff Walking with RL

In [1]:
import gymnasium as gym
import numpy as np
import scipy.optimize

## Make environment

In [2]:
env = gym.make("CliffWalking-v1")
for key in dir(env):
    if not key.startswith('_') and not callable(getattr(env, key)):
        print(key, getattr(env, key))
print(f'Type={type(env)}')

action_space Discrete(4)
env <PassiveEnvChecker<CliffWalkingEnv<CliffWalking-v1>>>
has_reset False
metadata {'render_modes': ['human', 'rgb_array', 'ansi'], 'render_fps': 4}
np_random Generator(PCG64)
np_random_seed 27109143815634670631414702805603456455
observation_space Discrete(48)
render_mode None
spec EnvSpec(id='CliffWalking-v1', entry_point='gymnasium.envs.toy_text.cliffwalking:CliffWalkingEnv', reward_threshold=None, nondeterministic=False, max_episode_steps=None, order_enforce=True, disable_env_checker=False, kwargs={}, namespace=None, name='CliffWalking', version=1, additional_wrappers=(), vector_entry_point=None)
unwrapped <CliffWalkingEnv<CliffWalking-v1>>
Type=<class 'gymnasium.wrappers.common.OrderEnforcing'>


Actions:
- 0 up
- 1 right
- 2 down
- 3 left

State space

 0  .. 11

 12 .. 23
 
 ........
 
 36 .. 47

Goal: from 36 to 47

State from 37 to 46 moves to 37

In [3]:
# State dynamics 
# action: [(probability, next state, reward, end episode indicator)]
env.unwrapped.P[14]

{0: [(1.0, np.int64(2), -1, False)],
 1: [(1.0, np.int64(15), -1, False)],
 2: [(1.0, np.int64(26), -1, False)],
 3: [(1.0, np.int64(13), -1, False)]}

## Policy evaluation

In [4]:
def evaluate_policy_bellman(env, policy, gamma=1., verbose=True):
    if verbose:
        print(f'policy={policy}')
    a, b = np.eye(env.unwrapped.nS), np.zeros(env.unwrapped.nS)
    for state in range(env.unwrapped.nS - 1):
        for action in range(env.unwrapped.nA):
            pi = policy[state][action]
            for p, next_state, reward, terminated in env.unwrapped.P[state][action]:
                a[state, next_state] -= (pi*gamma*p)
                b[state] += pi*reward*p
    v = np.linalg.solve(a, b)
    q = np.zeros((env.unwrapped.nS, env.unwrapped.nA))
    for state in range(env.unwrapped.nS - 1):
        for action in range(env.unwrapped.nA):
            for p, next_state, reward, terminated in env.unwrapped.P[state][action]:
                q[state, action] += reward + gamma * v[next_state] * p
    if verbose:
        print(f'state values={v}')
        print(f'action values={q}')
    return v, q

In [5]:
# Define start policy
policy = np.ones((env.unwrapped.nS, env.unwrapped.nA)) / env.unwrapped.nA

# define optimal policy
actions = np.ones(env.unwrapped.nS, dtype=int)
actions[36] = 0
actions[11::12] = 2
policy = np.eye(env.unwrapped.nA)[actions]

# random policy
policy = np.random.uniform(size=(env.unwrapped.nS, env.unwrapped.nA))
policy = policy / np.sum(policy, axis=1, keepdims=True)

In [6]:
state_values, action_values = evaluate_policy_bellman(env, policy)

policy=[[0.27094688 0.24353061 0.21872718 0.26679533]
 [0.20708974 0.3283094  0.22724327 0.23735758]
 [0.0009273  0.25130951 0.29715737 0.45060583]
 [0.27802196 0.28285854 0.30021371 0.13890579]
 [0.47587727 0.04966038 0.18123514 0.29322721]
 [0.17687781 0.31530639 0.36844749 0.13936831]
 [0.0224794  0.3702354  0.08409673 0.52318847]
 [0.135898   0.47430675 0.3422832  0.04751205]
 [0.32327704 0.19040756 0.17413325 0.31218215]
 [0.19050163 0.02965617 0.34567404 0.43416816]
 [0.23097599 0.34445323 0.19976438 0.2248064 ]
 [0.26123851 0.21281381 0.40066756 0.12528012]
 [0.33669664 0.05373809 0.35362393 0.25594134]
 [0.22129677 0.01219558 0.33750851 0.42899914]
 [0.28675098 0.27134088 0.18631286 0.25559528]
 [0.43217435 0.09314328 0.19518721 0.27949516]
 [0.0200588  0.18554184 0.4679209  0.32647846]
 [0.45751063 0.16964292 0.2944566  0.07838984]
 [0.24607659 0.34815439 0.13313675 0.27263226]
 [0.43463754 0.01222024 0.45614317 0.09699905]
 [0.35239258 0.29313409 0.26624007 0.08823326]
 [0.17

## Solve optimal values

In [16]:
def optimal_belman(env, gamma=1., verbose=True):
    nS, nA = env.unwrapped.nS, env.unwrapped.nA
    p = np.zeros((nS, nA, nS)) #  p[s][a][s']
    r= np.zeros((nS, nA)) #r[s][a]
    for state in range(nS - 1):
        for action in range(nA):
            for prob, next_state, reward, terminated in env.unwrapped.P[state][action]:
                p[state, action, next_state] += prob
                r[state, action] += reward*prob
                
    # Linear programming task
    # Variables: v[0], v[1], ..., v[nS-1].
    # Goal:minimize the sum of v[s].
    c = np.ones(nS)
    
    # limits
    # a_ub @ v <= b_ub
    p_flat = p.reshape(-1, nS)
    I_repeated = np.repeat(np.eye(nS), nA, axis=0)
    a_ub = gamma * p_flat - I_repeated
    b_ub = -r.reshape(-1)
    
    # No equality limits
    a_eq = np.zeros((0, nS))
    b_eq = np.zeros(0)
    
    # Bounds
    bounds = [(None, None) for _ in range(nS)]
    
    # Solution
    res = scipy.optimize.linprog(
        c,           # cost function
        a_ub, b_ub,  # inrqualities
        a_eq, b_eq,  # equalities
        bounds=bounds,
        method='interior-point'
    )
    
    v = res.x
    q = r + gamma * np.dot(p, v)
    
    if verbose:
        print("v*:")
        print(v)
        print("q*:")
        print(q)    
    return v, q

In [18]:
v_opt, q_opt = optimal_belman(env, gamma=1.0, verbose=True)

v*:
[-1.40000000e+01 -1.30000000e+01 -1.20000000e+01 -1.10000000e+01
 -1.00000000e+01 -9.00000000e+00 -8.00000000e+00 -7.00000000e+00
 -6.00000000e+00 -5.00000000e+00 -4.00000000e+00 -3.00000000e+00
 -1.30000000e+01 -1.20000000e+01 -1.10000000e+01 -1.00000000e+01
 -9.00000000e+00 -8.00000000e+00 -7.00000000e+00 -6.00000000e+00
 -5.00000000e+00 -4.00000000e+00 -3.00000000e+00 -2.00000000e+00
 -1.20000000e+01 -1.10000000e+01 -1.00000000e+01 -9.00000000e+00
 -8.00000000e+00 -7.00000000e+00 -6.00000000e+00 -5.00000000e+00
 -4.00000000e+00 -3.00000000e+00 -2.00000000e+00 -1.00000000e+00
 -1.30000000e+01 -1.20000000e+01 -1.10000000e+01 -1.00000000e+01
 -9.00000000e+00 -8.00000000e+00 -7.00000000e+00 -6.00000000e+00
 -5.00000000e+00 -4.00000000e+00 -9.99999999e-01  1.82302206e-11]
q*:
[[ -14.99999999  -13.99999999  -13.99999999  -14.99999999]
 [ -13.99999999  -13.          -13.          -14.99999999]
 [ -13.          -12.          -12.          -13.99999999]
 [ -12.          -11.          -11

C:\Users\alexe\AppData\Local\Temp\ipykernel_18620\1988825468.py:31: DeprecationWarning: `method='interior-point'` is deprecated and will be removed in SciPy 1.11.0. Please use one of the HiGHS solvers (e.g. `method='highs'`) in new code.
  res = scipy.optimize.linprog(


In [21]:
def get_optimal_policy(q):
    policy = np.argmax(q, axis=1) 
    return policy

def get_optimal_stochastic_policy(q):
    nS, nA = q.shape
    policy = np.zeros((nS, nA))
    optimal_actions = np.argmax(q, axis=1)
    policy[np.arange(nS), optimal_actions] = 1.0
    return policy

In [22]:
optimal_policy = get_optimal_policy(q_opt)

In [ ]:
v_eval, q_eval = evaluate_policy_bellman(env, get_optimal_stochastic_policy(q_opt), gamma=1.0)
print("Evaluation compare with optimal?", np.allclose(v_opt, v_eval))

policy=[[0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]]
state values=[-14. -13. -12. -11. -10.  -9.  -8.  -7.  -6.  -5.  -4.  -3. -13. -12.
 -11. -10.  -9.  -8.  -7.  -6.  -5.  -4.  -3.  -2. -12. -11. -10.  -9.
  -8.  -7.  -6.  -5.  -4.  -3.  -2.  -1. -13. -12. -11. -10.  -9.  -8.
  -7.  -6.  -5.  -4.  -1.   0.]
action values=